# 20 — Direction 1, step 6: replica caches and the baseline's null group

Per scenario, on a CPU:

1. **cache** (`hvac_controller/d1/cache.py`): spec, ROM, exogenous series, policy metadata and action spec, pickled
   once so that training and evaluation sessions never rebuild the ROM (≈ 1–4 min each). Also logs the Jacobi bound q
   and the float32 sweep count (must stay < 0.5 for the fast air solve).
2. **null group** (`hvac_controller/d1/evaluate.py`): the baseline K10 (`hvac_controller/baseline_k10.py`, 0.0185 public
   LB) for 30 days in the **original** simulator with seeds 0–3 (seed 0 plain, seeds ≥ 1 with σ = 1e-6 noise on the
   state observations), and in the NumPy **replica** with the same seeds (how close the replica scores K10's closed loop).

**MODE = "pilot"**: atlanta_2 only (step 6a), ≈ 15–20 min.
**MODE = "all"**: the 30 shipped scenarios, `N_SHARDS` sessions in parallel with `RUN_ID = 0 … N_SHARDS-1` (each takes
every N_SHARDS-th scenario), ≈ 2.5 h per session with 3 shards. A rerun of a shard skips jobs already done only if
their result files are attached; otherwise it starts over.

**Setup:** CPU; Internet on (clones the repo); attach the competition data.
**Output:** `/kaggle/working/d1_cache/caches/*.pkl` (attach this notebook's output to notebooks 21 and 22),
`/kaggle/working/d1_cache/eval/*.json` (one file per run), and `/kaggle/working/d1_cache_<mode>_<run>.zip` with the logs and
results (small; download it to read the log).

In [ ]:
REPO_URL = "https://github.com/CryAndRRich/hvac-controller"
REPO_REF = "direction1"
MODE = "pilot"            # "pilot" or "all"
RUN_ID, N_SHARDS = 0, 3   # all
PILOT = ["building_train_1/atlanta_2"]
SEEDS = 4                 # seed 0 plain + 3 noisy runs per scenario
N_PROCS = 4               # simulator runs in parallel (4 CPUs)
N_JOBS = 3                # cache builds in parallel

In [ ]:
import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

REPO = Path("/tmp/hvac-controller")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_REF], check=True)
COMMIT = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print(COMMIT)

SHIPPED = next(p for p in sorted(Path("/kaggle/input").rglob("training_scenarios"))
               if (p / "building_train_1" / "atlanta_2" / "config.gin").exists())
ROOT = REPO / "smart_buildings" / "training_scenarios"
shutil.copytree(SHIPPED, ROOT, dirs_exist_ok=True)
ALL = sorted(f"{p.parent.parent.name}/{p.parent.name}" for p in ROOT.glob("*/*/config.gin"))
SCENARIOS = PILOT if MODE == "pilot" else ALL[RUN_ID::N_SHARDS]
assert set(SCENARIOS) <= set(ALL), sorted(set(SCENARIOS) - set(ALL))
print(len(ALL), "scenarios;", len(SCENARIOS), "here:", SCENARIOS)

OUT = Path("/kaggle/working/d1_cache")
CACHES, EVAL, LOGS = OUT / "caches", OUT / "eval", OUT / "logs"
for d in (CACHES, EVAL, LOGS):
    d.mkdir(parents=True, exist_ok=True)
ENV = dict(os.environ, PYTHONPATH=str(REPO))

In [ ]:
# 1. Caches, N_JOBS at a time.
threads = str(max(1, (os.cpu_count() or 4) // N_JOBS))
env = dict(ENV, OMP_NUM_THREADS=threads, OPENBLAS_NUM_THREADS=threads, MKL_NUM_THREADS=threads)
t0 = time.time()
pending, running = list(SCENARIOS), []
while pending or running:
    while pending and len(running) < N_JOBS:
        scenario = pending.pop(0)
        tag = scenario.replace("/", "__")
        log = open(LOGS / f"cache__{tag}.txt", "w")
        cmd = [sys.executable, "-m", "hvac_controller.d1.cache", str(CACHES / f"{tag}.pkl"),
               "--config", str(ROOT / scenario / "config.gin")]
        running.append((tag, subprocess.Popen(cmd, cwd=REPO, env=env, stdout=log, stderr=subprocess.STDOUT), log))
    for item in list(running):
        tag, proc, log = item
        if proc.poll() is not None:
            log.close()
            running.remove(item)
            text = (LOGS / f"cache__{tag}.txt").read_text()
            print(f"[{time.time() - t0:5.0f}s] {tag} exit {proc.returncode}: {text.strip()[-1500:]}", flush=True)
    time.sleep(5)

In [ ]:
# 2. K10 in the original and in the replica, SEEDS runs each per scenario.
cmd = [sys.executable, "-m", "hvac_controller.d1.evaluate", str(EVAL), "--scenarios", ",".join(SCENARIOS),
       "--scenario-root", str(ROOT), "--cache-dir", str(CACHES), "--policies", "k10", "--seeds", str(SEEDS),
       "--where", "original,replica", "--n-procs", str(N_PROCS)]
with open(LOGS / "eval.txt", "w") as log:
    proc = subprocess.Popen(cmd, cwd=REPO, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="", flush=True)
        log.write(line)
print("exit", proc.wait())

In [ ]:
# 3. Summary and the small zip (logs, run results, cache infos).
infos = {p.stem: json.loads(p.read_text()) for p in sorted(CACHES.glob("*.json"))}
results = [json.loads(p.read_text()) for p in sorted(EVAL.glob("*.json"))]
(OUT / "summary.json").write_text(json.dumps({"commit": COMMIT, "mode": MODE, "run_id": RUN_ID, "caches": infos},
                                             indent=1, default=str))
name = f"/kaggle/working/d1_cache_{MODE}_{RUN_ID}.zip"
with zipfile.ZipFile(name, "w", zipfile.ZIP_DEFLATED) as z:
    for p in list(LOGS.glob("*")) + list(EVAL.glob("*.json")) + list(CACHES.glob("*.json")) + [OUT / "summary.json"]:
        z.write(p, p.relative_to(OUT.parent))
print("wrote", name)

for tag, i in infos.items():
    print(f"{tag:36s} Z={i['n_zones']:2d} V={i['n_vavs']:2d} A={i['n_ahus']} obs={i['n_obs']} r={i['order']} "
          f"h={i['h']:.2f} q={i['jacobi_q']:.3f} sweeps={i['sweeps_float32']} | ROM {i['rom_s']:.0f}s exo {i['exo_s']:.0f}s")
errors = [r for r in results if "error" in r]
for r in errors:
    print("ERROR", r["where"], r["scenario"], r["seed"], r["error"])
print(f"\n{'scenario':36s} {'original mean ± sd':>22} {'replica mean ± sd':>22} {'gap':>8}  original seeds")
by = {}
for r in results:
    if "error" not in r:
        by.setdefault((r["scenario"], r["where"]), []).append(r["score"])
for scenario in SCENARIOS:
    o, p = by.get((scenario, "original"), []), by.get((scenario, "replica"), [])
    fmt = lambda v: f"{sum(v) / len(v):.5f} ± {(sum((x - sum(v) / len(v)) ** 2 for x in v) / max(len(v) - 1, 1)) ** 0.5:.5f}" if v else "-"
    gap = f"{(sum(p) / len(p)) / (sum(o) / len(o)) - 1:+.2%}" if o and p else "-"
    print(f"{scenario:36s} {fmt(o):>22} {fmt(p):>22} {gap:>8}  {[round(x, 5) for x in o]}")
print(f"\nGATE 6.0: every cache has q < 0.5; every scenario has {SEEDS} original and {SEEDS} replica K10 runs; no errors:",
      all(i["jacobi_q"] < 0.5 for i in infos.values()) and not errors
      and all(len(by.get((s, w), [])) == SEEDS for s in SCENARIOS for w in ("original", "replica")))